In [1]:
!pip install -q openai-whisper
!pip install -q librosa
!pip install -q soundfile
!pip install -q gradio
!pip install -q nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.5 MB/s eta 0:00:00


In [2]:
import whisper
import librosa
import numpy as np
import re
import nltk
import gradio as gr

from collections import Counter

In [3]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper loaded successfully!")

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 166MiB/s]


Whisper loaded successfully!


In [5]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(audio_file)

    transcript = result["text"].strip()

    return transcript

In [6]:
filler_words = [
    "um",
    "uh",
    "hmm",
    "like",
    "actually",
    "basically",
    "you know",
    "so"
]


def analyze_text(text):

    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)

    filler_count = 0

    found_fillers = []

    for filler in filler_words:

        count = len(
            re.findall(
                r'\b' + re.escape(filler) + r'\b',
                text.lower()
            )
        )

        if count > 0:
            filler_count += count
            found_fillers.append(
                f"{filler}: {count}"
            )

    return {
        "word_count": word_count,
        "filler_count": filler_count,
        "fillers": found_fillers
    }

In [7]:
def analyze_speech(audio_file, transcript):

    audio, sample_rate = librosa.load(
        audio_file,
        sr=None
    )

    duration = librosa.get_duration(
        y=audio,
        sr=sample_rate
    )

    words = len(
        re.findall(
            r'\b\w+\b',
            transcript
        )
    )

    if duration > 0:

        speaking_rate = (
            words / duration
        ) * 60

    else:

        speaking_rate = 0

    return {
        "duration": round(duration, 2),
        "speaking_rate": round(
            speaking_rate,
            2
        )
    }

In [8]:
def calculate_score(
    word_count,
    filler_count,
    speaking_rate
):

    score = 100

    # Penalize excessive filler words
    score -= filler_count * 3

    # Speaking rate evaluation
    if speaking_rate < 90:
        score -= 10

    elif speaking_rate > 180:
        score -= 10

    # Avoid negative score
    score = max(score, 0)

    return score

In [9]:
def generate_feedback(
    filler_count,
    speaking_rate
):

    feedback = []

    if filler_count == 0:
        feedback.append(
            "Excellent! No major filler words detected."
        )

    elif filler_count <= 3:
        feedback.append(
            "Good control of filler words."
        )

    else:
        feedback.append(
            "Try to reduce filler words."
        )

    if speaking_rate < 90:
        feedback.append(
            "Try speaking slightly faster."
        )

    elif speaking_rate > 180:
        feedback.append(
            "Try speaking more slowly and clearly."
        )

    else:
        feedback.append(
            "Your speaking rate is good."
        )

    return feedback

In [10]:
def interview_analyzer(audio_file):

    if audio_file is None:
        return (
            "Please record or upload an interview answer.",
            "",
            "",
            "",
            "",
            ""
        )

    # 1. Speech to Text
    transcript = speech_to_text(audio_file)

    if not transcript:
        return (
            "Could not understand the speech.",
            "",
            "",
            "",
            "",
            ""
        )

    # 2. Text Analysis
    text_result = analyze_text(transcript)

    # 3. Speech Analysis
    speech_result = analyze_speech(
        audio_file,
        transcript
    )

    # 4. Calculate Score
    score = calculate_score(
        text_result["word_count"],
        text_result["filler_count"],
        speech_result["speaking_rate"]
    )

    # 5. Generate Feedback
    feedback = generate_feedback(
        text_result["filler_count"],
        speech_result["speaking_rate"]
    )

    # Format results
    filler_text = ", ".join(
        text_result["fillers"]
    )

    feedback_text = "\n".join(
        "• " + item for item in feedback
    )

    speech_info = (
        f"Duration: {speech_result['duration']} seconds\n"
        f"Speaking Rate: {speech_result['speaking_rate']} words/minute"
    )

    text_info = (
        f"Word Count: {text_result['word_count']}\n"
        f"Filler Words: {text_result['filler_count']}\n"
        f"Detected Fillers: {filler_text if filler_text else 'None'}"
    )

    score_info = f"Interview Score: {score}/100"

    return (
        transcript,
        text_info,
        speech_info,
        score_info,
        feedback_text,
        filler_text
    )

In [12]:
import gradio as gr

with gr.Blocks(title="Interview Voice Analyzer") as app:

    gr.Markdown("""
    # 🎤 Interview Voice Analyzer

    ### Analyze your interview answer using Speech and Text Analysis
    """)

    audio_input = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎤 Record / Upload Interview Answer"
    )

    analyze_button = gr.Button(
        "🔍 Analyze Interview",
        variant="primary"
    )

    transcript_output = gr.Textbox(
        label="📝 Recognized Speech",
        lines=6
    )

    text_output = gr.Textbox(
        label="📊 Text Analysis",
        lines=4
    )

    speech_output = gr.Textbox(
        label="🎙️ Speech Analysis",
        lines=3
    )

    score_output = gr.Textbox(
        label="⭐ Interview Score"
    )

    feedback_output = gr.Textbox(
        label="💡 Feedback",
        lines=5
    )

    analyze_button.click(
        fn=interview_analyzer,
        inputs=audio_input,
        outputs=[
            transcript_output,
            text_output,
            speech_output,
            score_output,
            feedback_output,
            gr.Textbox(visible=False)
        ]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e55f7c675017578b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
